# TestBED: Introduction and Examples
Author: Gavin Kerrigan (kerrigan@stats.ox.ac.uk)

Date: 2026 April 17

This notebook is a short tutorial and basic overview of some of the functionality of TestBED.

___
## Introduction


TestBED is a library for Bayesian experimental design implemented in PyTorch. The two core components to any BED problem are:
- A `Task`, which represents the problem we are applying BED to
- A `Designer`, which is our strategy for producing experimental designs for our `Task`.

TestBED provides reference implementations for several common choices of `Task`s and `Designer`s, as well as some of the scaffolding to glue these together in an end-to-end BED loop. TestBED is highly extensible, and facilitates the development ot new, custom `Task`s and `Designer`s. 

___
## Tasks 

The first object we will create is a `Task`. This is the object which implements the model which you are doing experimental design for. That is, it implements a prior $p(\theta)$ and likelihood $p(y \mid \theta, \xi)$. 

TestBED supports both explicit likelihoods and implicit likelihoods (i.e., those for which we can only sample $y \sim p(y \mid \theta, \xi)$ but not evaluate this likelihood). TestBED also supports problems with non-sationary likelihoods, i.e., those for which $p(y \mid \theta, \xi, h_t) \neq p(y \mid \theta, \xi)$ for some experimental history $h_t$. We will not cover these more advanced cases in this introduction.

The classic example is the `LocationFinding` task. Creating the task as follows will create an instance with some default hyperparameters defining the likelihood, as well as a standard Gaussian prior.

In [1]:
import torch
from testbed.tasks import LocationFinding

n_sources = 4
dim_theta = 2
task = LocationFinding(n_sources, dim_theta)
print(task)


LocationFinding(n_sources=4, dim_theta=2, dim_y=1, base_signal=0.1, max_signal=0.0001, noise_scale=0.5, exponent=2.0)


The only thing a specific `task` needs to have implemented are:
- A `task.prior` object which supports `task.prior.sample(n_samples)`
- Methods `task._sample_outcome(theta, design)` and, optionally for explicit methods, `task._outcome_logp(y, theta, design)`

Once these are implemented, TestBED can handle the rest. For example, some handy things you can do include:
- `task.sample_prior(n)` will sample $\theta_i \sim p(\theta)$.
- `task.sample_joint(design, n)` will draw n samples $(y_i, \theta_i) \sim p(y, \theta \mid \xi)$ from the joint using a fixed design
- `task.outcome_logp(y, theta, design)` will evaluate $\log p(y_i \mid \theta_i, \xi_i)$.
- `task.history_logp(y_history, xi_history, theta)` will evaluate $\sum_t \log p(y_t \mid \theta, \xi_t)$

In [4]:
# Demonstrating broadcasting semantics
# Here, design has no explicit batch dim and so samples don't either

design = torch.randn((dim_theta))
y, theta = task.sample_joint(design, n_samples=100)

print(f'y: {y.shape} theta: {theta.shape}')

y: torch.Size([100, 1]) theta: torch.Size([100, 4, 2])


In [15]:
# Here, design has an explicit batch dim of 1
# and so samples are of shape [n_samples, *batch_dimensions, *event_dimensions]

design = torch.randn((1, dim_theta))
y, theta = task.sample_joint(design, n_samples=100)

print(f'y: {y.shape} theta: {theta.shape}')

y: torch.Size([100, 1, 1]) theta: torch.Size([100, 1, 4, 2])


In [16]:
b = 100

# Supports b=1 (or no batch shape at all) for any of these and will automatically broadcast
y = torch.randn((b, 1))
theta = torch.randn((b, n_sources, dim_theta))
design = torch.randn((b, b, dim_theta))

# Assumes inputs are all paired i.e. (y_i, theta_i, xi_i)
logp = task.outcome_logp(y, theta, design)
print(f'logp: {logp.shape}')

logp: torch.Size([100, 100])


In [17]:
b = 100
T = 10

y = torch.randn((b, T, 1))
theta = torch.randn((b, n_sources, dim_theta))
design = torch.randn((b, T, dim_theta))

# Assumes inputs y, design are a rollout from a Designer of shape [b, T, *event_shape]
# where xi is fixed [b, *event_shape_design] throughout each of the b independent rollouts 
logp = task.history_logp(y, design, theta)
print(f'logp: {logp.shape}')

logp: torch.Size([100])


An aside on `Task` priors: as mentioned above, the only requirement is that a prior has a `prior.sample(n_samples)` method. Internally, this is enforced by inheriting from the `testbed.core.distributions.Distribution` class. For convenience, we also provide a thin `TorchDistribution` wrapper to help with constructing priors directly from PyTorch distribution objects, as in the following example.

In [18]:
from testbed.core.distributions import TorchDistribution
from torch.distributions.multivariate_normal import MultivariateNormal

n_sources = 4
dim_theta = 2

# Equivalent to tasks.location_findng.LocationFindingPrior
torch_prior = TorchDistribution(
    MultivariateNormal(torch.zeros(n_sources, dim_theta),
                       torch.eye(dim_theta).expand(n_sources, -1, -1)))


task = LocationFinding(n_sources, dim_theta, prior=torch_prior)

prior_samples = task.sample_prior(100)
print(f'prior_samples: {prior_samples.shape}')


prior_samples: torch.Size([100, 4, 2])


___
## Designers

Our second ingredient is a `Designer`. Intuitively, these are objects that propose new designs throughout an experimental design rollout. A `Designer` will generally be constructed with a `Task` and an experiment rollout length `T`, plus any additional settings needed. 

TestBED includes two `Designers` which already are likely to cover many use cases:

- `RandomDesigner` which randomly selects designs from some pre-specified distribution
- `PolicyDesigner`, which uses a neural network (of your choosing) to propose new designs



In [19]:
from testbed.designers import RandomDesigner

# Also demonstrates how to use devices
device = 'cuda' if torch.cuda.is_available() else 'cpu'
task = LocationFinding(n_sources, dim_theta)
task.to(device)

# Create a RandomDesigner that samples from a standard Gaussian
design_prior = TorchDistribution(
    torch.distributions.MultivariateNormal(
        torch.zeros(dim_theta, device=device),
        torch.eye(dim_theta, device=device),
    ))

T = 10
random_designer = RandomDesigner(task, T, design_prior)

# Propose a one-step-ahead design, conditioned on (y_history, xi_history)
# Note that t=0 corresponds to a null/empty history
b = 5
y_history, design_history = random_designer.empty_history(b) 
# Equivalent to:
# y_history = torch.empty((b, t, 1))                              # [b, 0, dim_y]
# design_history = torch.empty((b, t, *task.event_shape_design))  # [b, 0, dim_theta] for location finding

design = random_designer.propose_design(y_history, design_history)
print(f'design: {design.shape}')

# Roll-out the random_designer for the full T=10 steps, y, xi ~ p(h_T | theta_0)
# Note that this conditions on a true value of theta0
theta0 = torch.randn((b, *task.event_shape_theta), device=device) # [b, num_sources, dim_theta] for location finding
y_obs, design_obs = random_designer.rollout(theta0)    # iteratively calls random_designer.propose_design
print(f'y_obs: {y_obs.shape}, design_obs: {design_obs.shape}')

design: torch.Size([5, 2])
y_obs: torch.Size([5, 10, 1]), design_obs: torch.Size([5, 10, 2])


Randomly selecting designs isn't very fun. Let's instead use a neural network based policy to propose designs. In TestBED, we provide some standard reference implementations of various policy architectures under `testbed/design_nets`. Let's import a basic DeepSets-style model here. 

In [20]:
from testbed.designers import PolicyDesigner
from testbed.design_nets import DADNet_v2

policy_net = DADNet_v2(dim_y=1, dim_xi=dim_theta)
policy_net.to(device)

policy_designer = PolicyDesigner(task, T, policy_net)

# In exactly the same way before, we can roll-out our policy-net based designer
theta0 = torch.randn((b, *task.event_shape_theta), device=device) # [b, num_sources, dim_theta] for location finding
y_obs, xi_obs = policy_designer.rollout(theta0)    # iteratively calls random_designer.propose_design
print(f'y_obs: {y_obs.shape}, xi_obs: {xi_obs.shape}')


y_obs: torch.Size([5, 10, 1]), xi_obs: torch.Size([5, 10, 2])


___

## Objectives

Finally, we need some way of training and evaluating our policies. TestBED currently implements SPCE and SNMC [Gav: Cite]. Evaluating (and scaling) the calculation of SPCE/SNMC can be expensive in both time and memory, so TestBED aims to provide an efficient implementation of these.

Both objectives require you to specify the `task`, since estimating either objective requires draws from the model. 

In [24]:
from testbed.design_objectives import SPCE, SNMC

spce = SPCE(task, n_contrastive=1_024)
snmc = SNMC(task, n_contrastive=1_024)

with torch.no_grad():
    spce_val = spce.estimate_gain(y_obs, xi_obs, theta0)
    snmc_val = snmc.estimate_gain(y_obs, xi_obs, theta0)
print(f'spce={spce_val}, snmc={snmc_val}')

spce=2.8547840118408203, snmc=2.9605319499969482


___
## Putting it All Together

Let's now take our ingredients and put them together into a basic policy training loop. This will train a policy on `LocationFinding` for a few steps by doing gradient ascent on SPCE. 

For a more fleshed-out recipe for policy training, see `testbed/recipes/TODO`.

From here, we hope that you are able to extend TestBED to your needs. Happy hacking!

In [25]:
import torch
from torch.optim import Adam

from testbed.tasks import LocationFinding
from testbed.designers import PolicyDesigner
from testbed.design_nets import DADNet_v2
from testbed.design_objectives import SPCE

def train():
    torch.manual_seed(0)
    
    n_sources = 1
    dim_theta = 2
    T = 10

    # Number of inner/outer samples in SPCE estimation
    n_outer = 1024
    n_inner = 1024

    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    task = LocationFinding(n_sources, dim_theta)
    task.to(device)
    spce = SPCE(task, n_contrastive=n_inner) 
    designer = PolicyDesigner(task,
                               T, 
                               DADNet_v2().to(device))

    lr=1e-4
    optim = Adam(designer.policy_net.parameters(), lr=lr)

    n_step = 100
    for step in range(n_step):
        # theta0 ~ p(theta)
        theta0 = task.sample_prior(n_outer)
        # y_1:T, xi_1:T ~ p(h_T | policy, theta0)
        y_obs, design_obs = designer.rollout(theta0)

        # SPCE is a lower bound which we try to maximize
        spce_val = spce.estimate_gain(y_obs, design_obs, theta0)
        loss = -1. * spce_val
        
        optim.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(designer.policy_net.parameters(), 1.0)
        optim.step()

        if (step + 1) % 10 == 0:
            print(f' Step [{step+1}/{n_step}] | SPCE: {spce_val}')
    
    return designer

In [26]:
designer = train()

 Step [10/100] | SPCE: 2.491814613342285
 Step [20/100] | SPCE: 2.583779811859131
 Step [30/100] | SPCE: 2.64774227142334
 Step [40/100] | SPCE: 2.812873601913452
 Step [50/100] | SPCE: 2.8668692111968994
 Step [60/100] | SPCE: 2.949721574783325
 Step [70/100] | SPCE: 2.9690651893615723
 Step [80/100] | SPCE: 2.869194984436035
 Step [90/100] | SPCE: 3.0554611682891846
 Step [100/100] | SPCE: 3.0354621410369873
